# 🎭 AvatarForcing — End-to-End RunPod RTX 5090 Notebook
One-Step Streaming Talking Avatars — run all cells top to bottom.

## 🖥️ Step 0 — GPU Check

In [ ]:
import subprocess, sys, os

def run(cmd, check=True, **kwargs):
    print(f'>>> {cmd}')
    result = subprocess.run(cmd, shell=True, text=True, **kwargs)
    if check and result.returncode != 0:
        raise RuntimeError(f'Command failed (exit {result.returncode}): {cmd}')
    return result

run('nvidia-smi')
run('nvcc --version')

## 🐍 Step 1 — Find or Install Conda

In [ ]:
import shutil, os, subprocess

CONDA_SEARCH_PATHS = [
    '/opt/conda/bin/conda',
    '/root/miniconda3/bin/conda',
    '/root/anaconda3/bin/conda',
    '/usr/local/conda/bin/conda',
]

CONDA_BIN = shutil.which('conda')
if CONDA_BIN is None:
    for p in CONDA_SEARCH_PATHS:
        if os.path.isfile(p):
            CONDA_BIN = p
            break

if CONDA_BIN:
    print(f'✅ Found conda at: {CONDA_BIN}')
    subprocess.run(f'{CONDA_BIN} --version', shell=True)
else:
    print('Installing Miniconda...')
    for c in [
        'curl -fsSL https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -o /tmp/miniconda.sh',
        'bash /tmp/miniconda.sh -b -p /root/miniconda3',
    ]:
        run(c)
    CONDA_BIN = '/root/miniconda3/bin/conda'

CONDA_DIR = os.path.dirname(os.path.dirname(CONDA_BIN))
print(f'CONDA_DIR = {CONDA_DIR}')

## 📁 Step 2 — Clone Repository

In [ ]:
REPO_DIR = '/workspace/AvatarForcing-inference-v1'

if not os.path.isdir(REPO_DIR):
    run(f'git clone https://github.com/MoshiHead/AvatarForcing-inference-v1.git {REPO_DIR}')
else:
    print(f'✅ Repo already exists at {REPO_DIR}')

os.chdir(REPO_DIR)
run('ls -la')

## 🌿 Step 3 — Patch environment.yml & Create Conda Env

**Three patches applied to `environment.yml` before creation:**
1. `defaults` → `conda-forge` — fixes `CondaToSNonInteractiveError`
2. Remove `torch / torchvision / torchaudio` pip lines — `+cu121` tag only exists on the PyTorch index, not PyPI
3. Remove `flash-attn` pip line — needs torch already installed to compile
4. Remove `nvidia-*` and `cuda-*` pip lines — CUDA packages also require special index
5. Remove `triton` pip line — depends on torch

All removed packages are installed correctly in **Step 5**.

In [ ]:
# 3a — Accept Anaconda TOS
run(f'{CONDA_BIN} tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main', check=False)
run(f'{CONDA_BIN} tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r', check=False)
print('✅ TOS accepted.')

In [ ]:
import re, shutil as _sh

yml_path = f'{REPO_DIR}/environment.yml'
yml_bak  = f'{REPO_DIR}/environment.yml.bak'

# Always restore from original backup so re-running is idempotent
if os.path.exists(yml_bak):
    _sh.copy(yml_bak, yml_path)
    print('Restored original from backup.')
else:
    _sh.copy(yml_path, yml_bak)
    print(f'Backup saved: {yml_bak}')

with open(yml_path) as fh:
    content = fh.read()

# --- Patch 1: defaults → conda-forge ---
content = content.replace('  - defaults', '  - conda-forge')

# --- Patches 2-5: remove packages that need special index or pre-installed torch ---
# Pattern: match any pip list line containing these package prefixes
REMOVE_PREFIXES = [
    'flash.attn',   # needs torch to build
    'torch==',      # +cu121 not on PyPI
    'torchvision',  # +cu121 not on PyPI
    'torchaudio',   # +cu121 not on PyPI
    'torchmetrics', # depends on torch version
    'nvidia-',      # CUDA packages need torch index
    'cuda-',        # CUDA runtime packages
    'triton==',     # needs torch
    'pytorch-lightning', # depends on torch
]

removed = []
lines_out = []
for line in content.splitlines(keepends=True):
    stripped = line.strip().lstrip('-').strip()
    matched = any(stripped.lower().startswith(p.lower()) for p in REMOVE_PREFIXES)
    if matched:
        removed.append(stripped.split('==')[0])
    else:
        lines_out.append(line)

content = ''.join(lines_out)

with open(yml_path, 'w') as fh:
    fh.write(content)

print(f'✅ environment.yml patched — removed {len(removed)} problematic pip packages:')
for p in removed:
    print(f'   - {p}')

In [ ]:
# 3c — Create conda env
result = subprocess.run(f'{CONDA_BIN} env list', shell=True, text=True, capture_output=True)
print(result.stdout)

if 'avatarforcing' in result.stdout:
    print("✅ Conda env 'avatarforcing' already exists. Skipping.")
else:
    print('Creating conda env (may take 10–20 min)...')
    run(f'{CONDA_BIN} env create -f {REPO_DIR}/environment.yml')
    print('✅ Conda env created!')

In [ ]:
# Helper: run a command inside the avatarforcing env
def run_in_env(cmd, **kwargs):
    return run(f'{CONDA_BIN} run -n avatarforcing --no-capture-output {cmd}', **kwargs)

run_in_env('python --version')
run_in_env('pip --version')

## 🔧 Step 4 — Install System Dependencies (FFmpeg)

In [ ]:
run('apt-get update -qq && apt-get install -y ffmpeg libsm6 libxext6 -qq')
run('ffmpeg -version 2>&1 | head -3')

## ⚡ Step 5 — Install PyTorch (cu126) + flash-attn for RTX 5090

RTX 5090 = Blackwell / SM_100.  
We install `cu126` PyTorch wheels (from the official PyTorch index),  
then install **flash-attn via a pre-built wheel** (seconds, not hours!).

| Component | Version |
|---|---|
| CUDA | 12.6 (cu126) |
| PyTorch | 2.6.0 |
| Python | 3.11 |
| flash-attn | 2.7.4 (pre-built binary) |

In [ ]:
# 🔧 FIX — Replace torch 2.6.0 with nightly that supports RTX 5090 (sm_120)
print('Uninstalling old torch stack...')
run_in_env('pip uninstall -y torch torchvision torchaudio')

print('Installing PyTorch nightly cu128 (sm_120 / Blackwell support)...')
run_in_env(
    'pip install --pre torch torchvision torchaudio '
    '--index-url https://download.pytorch.org/whl/nightly/cu128'
)
print('✅ Done. Verifying...')
run_in_env(
    'python -c "'
    'import torch; '
    'print(torch.__version__); '
    'print(torch.cuda.get_device_capability()); '
    'x = torch.tensor([1.0]).cuda(); '
    'print(x * 2, \'GPU OK\')'
    '"'
)

In [ ]:
# 5b — Install pytorch-lightning and torchmetrics (compatible with torch 2.6.0)
run_in_env('pip install pytorch-lightning torchmetrics -q')
print('✅ pytorch-lightning + torchmetrics installed.')

In [ ]:
# 5c — Verify torch sees the GPU
run_in_env('python -c "import torch; print(torch.__version__, torch.version.cuda, torch.cuda.is_available(), torch.cuda.get_device_name(0))"')

In [ ]:
print('Installing required packages...')

run_in_env(
    'pip install omegaconf peft easydict diffusers ftfy lmdb pandas opencv-python decord soundfile einops av'
)

print('✅ All packages installed.')

## 📦 Step 6 — Install huggingface_hub & (Optional) Login

In [ ]:
run_in_env("pip install -q 'huggingface_hub[cli]'")

HF_TOKEN = ''  # paste token if needed; all 3 models are public
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
    run_in_env(f'huggingface-cli login --token {HF_TOKEN}')
else:
    print('ℹ️  No HF token — public access is fine.')

## ⬇️ Step 7 — Download Models

| Model | Repo | Size |
|---|---|---|
| Wan2.1-T2V-1.3B | `Wan-AI/Wan2.1-T2V-1.3B` | ~5 GB |
| wav2vec2-base-960h | `facebook/wav2vec2-base-960h` | ~360 MB |
| AvatarForcing | `lycui/AvatarForcing` | ~2 GB |

In [ ]:
WAN_DIR     = f'{REPO_DIR}/wan_models/Wan2.1-T2V-1.3B'
WAV2VEC_DIR = f'{REPO_DIR}/wan_models/wav2vec2-base-960h'
CKPT_DIR    = f'{REPO_DIR}/checkpoints'
for d in [WAN_DIR, WAV2VEC_DIR, CKPT_DIR]:
    os.makedirs(d, exist_ok=True)
print('Directories ready.')

In [ ]:
if not any(os.scandir(WAN_DIR)):
    run_in_env(f'hf download Wan-AI/Wan2.1-T2V-1.3B --local-dir {WAN_DIR}')
else:
    print(f'✅ Wan model already at {WAN_DIR}')

In [ ]:
if not any(os.scandir(WAV2VEC_DIR)):
    run_in_env(f'hf download facebook/wav2vec2-base-960h --local-dir {WAV2VEC_DIR}')
else:
    print(f'✅ wav2vec2 already at {WAV2VEC_DIR}')

In [ ]:
pt_files = [f for f in os.listdir(CKPT_DIR) if f.endswith('.pt')]
if not pt_files:
    run_in_env(f'hf download lycui/AvatarForcing --local-dir {CKPT_DIR}')
else:
    print(f'✅ Checkpoints present: {pt_files}')

In [ ]:
for d in [WAN_DIR, WAV2VEC_DIR, CKPT_DIR]:
    print(f'📂 {d}  ({len(os.listdir(d))} files)')

## 🔍 Step 8 — Locate Checkpoint File

In [ ]:
import glob

pt_files = list(set(
    glob.glob(f'{CKPT_DIR}/**/*.pt', recursive=True) +
    glob.glob(f'{CKPT_DIR}/*.pt')
))
for f in pt_files:
    print(f'  {f}  ({os.path.getsize(f)/1e6:.1f} MB)')

preferred = [f for f in pt_files if os.path.basename(f) == 'model.pt']
CHECKPOINT_PATH = preferred[0] if preferred else (pt_files[0] if pt_files else None)
print(f'\n✅ Using: {CHECKPOINT_PATH}')

## 🎬 Step 9 — Prepare Input Data

In [ ]:
USE_EXAMPLE = True  # set False to use custom image/audio

if USE_EXAMPLE:
    DATA_PATH = f'{REPO_DIR}/example/promot.txt'
    print('Using bundled example.')
    print(open(DATA_PATH).read())
else:
    IMAGE_PATH  = '/workspace/my_image.jpg'
    AUDIO_PATH  = '/workspace/my_audio.wav'
    PROMPT_TEXT = 'A person speaks naturally facing the camera, clear lip movements, neutral background.'
    DATA_PATH = f'{REPO_DIR}/custom_prompt.txt'
    with open(DATA_PATH, 'w') as fh:
        fh.write(f'{IMAGE_PATH} {AUDIO_PATH} "{PROMPT_TEXT}"\n')
    print(f'Created: {DATA_PATH}')

In [ ]:
from IPython.display import Image as IPImage, display
parts = open(DATA_PATH).readline().strip().split()
img_path, audio_path = parts[0], parts[1]
print(f'Image : {img_path}\nAudio : {audio_path}')
if os.path.exists(img_path):
    display(IPImage(filename=img_path, width=300))

## ⚙️ Step 10 — Configure Inference

In [ ]:
CONFIG_PATH   = f'{REPO_DIR}/configs/avatarforcing.yaml'
OUTPUT_FOLDER = f'{REPO_DIR}/outputs'

# (NUM_OUTPUT_FRAMES - 1) must be divisible by 4
# Valid: 21 (0.8s), 81 (3.2s), 125 (4.9s), 225 (8.9s)
NUM_OUTPUT_FRAMES = 225
SEED        = 42
NUM_SAMPLES = 1
USE_EMA     = False

os.makedirs(OUTPUT_FOLDER, exist_ok=True)
assert (NUM_OUTPUT_FRAMES - 1) % 4 == 0, '(NUM_OUTPUT_FRAMES-1) must be divisible by 4!'
print(f'✅ {NUM_OUTPUT_FRAMES} frames (~{(NUM_OUTPUT_FRAMES-1)/25:.1f}s @ 25 FPS)')

## 🚀 Step 11 — Run Inference

In [ ]:
wav2vec_path = '/workspace/AvatarForcing-inference-v1/wan/models/wav2vec.py'

with open(wav2vec_path, 'r') as f:
    src = f.read()

# Remove both occurrences of the forbidden line
fixed = src.replace('        self.config.output_attentions = True\n', '')

if fixed == src:
    print('⚠️  Line not found — check file manually')
else:
    with open(wav2vec_path, 'w') as f:
        f.write(fixed)
    count = src.count('        self.config.output_attentions = True')
    print(f'✅ Removed {count} occurrence(s) of self.config.output_attentions = True')
    print('   File saved. Re-run inference now.')

In [ ]:
attn_path = '/workspace/AvatarForcing-inference-v1/wan/modules/attention.py'

new_src = '''# Copyright 2024-2025 The Alibaba Wan Team Authors. All rights reserved.
# flash-attn replaced with torch SDPA for RTX 5090 (sm_120) compatibility
import warnings
import torch
import torch.nn.functional as F

FLASH_ATTN_3_AVAILABLE = False
FLASH_ATTN_2_AVAILABLE = False

__all__ = [
    'flash_attention',
    'attention',
]

def flash_attention(
    q, k, v,
    q_lens=None,
    k_lens=None,
    dropout_p=0.,
    softmax_scale=None,
    q_scale=None,
    causal=False,
    window_size=(-1, -1),
    deterministic=False,
    dtype=torch.bfloat16,
    version=None,
):
    """
    Drop-in replacement using PyTorch SDPA (RTX 5090 / sm_120 compatible).
    q: [B, Lq, Nq, C1]
    k: [B, Lk, Nk, C1]
    v: [B, Lk, Nk, C2]
    """
    half_dtypes = (torch.float16, torch.bfloat16)
    assert dtype in half_dtypes
    b, lq, lk, out_dtype = q.size(0), q.size(1), k.size(1), q.dtype

    def half(x):
        return x if x.dtype in half_dtypes else x.to(dtype)

    # Handle variable-length sequences by padding/masking
    if q_lens is not None or k_lens is not None:
        warnings.warn(
            'Padding mask is disabled when using scaled_dot_product_attention. '
            'It can have a significant impact on performance.'
        )

    if q_scale is not None:
        q = q * q_scale

    # q/k/v: [B, L, N, C] -> [B, N, L, C] for SDPA
    q = half(q).transpose(1, 2)
    k = half(k).transpose(1, 2)
    v = half(v).transpose(1, 2)

    scale = softmax_scale if softmax_scale is not None else q.shape[-1] ** -0.5

    out = F.scaled_dot_product_attention(
        q, k, v,
        scale=scale,
        dropout_p=dropout_p if torch.is_grad_enabled() else 0.0,
        is_causal=causal,
    )
    # [B, N, L, C] -> [B, L, N, C]
    return out.transpose(1, 2).to(out_dtype)


def attention(
    q, k, v,
    q_lens=None,
    k_lens=None,
    dropout_p=0.,
    softmax_scale=None,
    q_scale=None,
    causal=False,
    window_size=(-1, -1),
    deterministic=False,
    dtype=torch.bfloat16,
    fa_version=None,
):
    return flash_attention(
        q=q, k=k, v=v,
        q_lens=q_lens, k_lens=k_lens,
        dropout_p=dropout_p,
        softmax_scale=softmax_scale,
        q_scale=q_scale,
        causal=causal,
        window_size=window_size,
        deterministic=deterministic,
        dtype=dtype,
        version=fa_version,
    )
'''

with open(attn_path, 'w') as f:
    f.write(new_src)
print('✅ attention.py fully rewritten — both flash_attention and attention use SDPA.')

In [ ]:
ema_flag = '--use_ema' if USE_EMA else ''
cmd = (
    f'python {REPO_DIR}/inference.py '
    f'--config_path {CONFIG_PATH} '
    f'--output_folder {OUTPUT_FOLDER} '
    f'--checkpoint_path {CHECKPOINT_PATH} '
    f'--data_path {DATA_PATH} '
    f'--num_output_frames {NUM_OUTPUT_FRAMES} '
    f'--seed {SEED} --num_samples {NUM_SAMPLES} '
    f'--i2v {ema_flag}'
)
print('Command:', cmd)
run_in_env(cmd)

## 🎥 Step 12 — Display Output Video

In [ ]:
from IPython.display import Video, display
videos = sorted(glob.glob(f'{OUTPUT_FOLDER}/*.mp4'), key=os.path.getmtime, reverse=True)
if videos:
    print(f'Latest: {videos[0]}  ({os.path.getsize(videos[0])/1e6:.1f} MB)')
    display(Video(videos[0], embed=True, width=832, height=480))
    for v in videos:
        print(f'  {v}')
else:
    print('⚠️  No output videos found.')

## 🩺 Troubleshooting

| Issue | Fix |
|---|---|
| `conda: not found` | Step 1 auto-installs Miniconda |
| `CondaToSNonInteractiveError` | Step 3a accepts TOS |
| `torch==2.5.1+cu121 not found` | Fixed: torch removed from yml; installed via PyTorch index in Step 5 |
| `flash-attn build takes hours` | Fixed: pre-built wheel installed in Step 5d (seconds) |
| `CUDA not available` | Step 5a installs cu126 wheels |
| OOM error | Reduce `NUM_OUTPUT_FRAMES` to 81 |
| `(N-1) % 4 != 0` | Use: 21, 25, 81, 125, 225 |
| No `.pt` checkpoint | Re-run Step 7 download cell |